# Pandas — «Hazlo tú» (SOLUCIONES DOCENTES)

**Nombre y apellidos:**  

**Fecha:**  

> Entrega este cuaderno en Moodle **al final de la clase**. Entregarlo acredita **asistencia**; que esté **completo** acredita **atención y seguimiento**. La valoración es **hecho / no hecho**: no se corrige acierto.


## 0. Preparación

Ejecuta esta celda **una vez**. No la modifiques: genera los datos que usarás en todos los bloques.


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1000
empleados = pd.DataFrame({
    "id": np.arange(1, n + 1),
    "edad": rng.integers(18, 65, size=n),
    "salario": rng.normal(2500, 800, size=n).round(2),
    "experiencia": np.clip(rng.integers(18, 65, size=n) - rng.integers(18, 30, size=n), 0, None),
    "horas": rng.normal(40, 5, size=n).round(1),
    "nivel": rng.integers(0, 5, size=n),
    "satisfaccion": rng.normal(7, 2, size=n).round(2),
})
empleados.loc[rng.choice(n, 10, replace=False), "salario"] *= 4
empleados.loc[rng.choice(n, 15, replace=False), "satisfaccion"] = np.nan

empleados_idx = empleados.set_index("id")
print("Filas y columnas:", empleados.shape)
print(empleados.head(3))


## Bloque 1 — Series y DataFrame

1. Selecciona las columnas `edad` y `salario` y muestra las 3 primeras filas con `head(3)`.
2. Imprime la forma (`shape`) y los tipos (`dtypes`) de esa selección.

**Resultado esperado:** 3 filas × 2 columnas; `edad` entera y `salario` decimal.


In [ ]:
sel = empleados[["edad", "salario"]]
print(sel.head(3))
print(sel.shape)
print(sel.dtypes)


## Bloque 2 — Selección y filtros

Con `empleados_idx` (índice = `id`):

1. Obtén `edad` y `salario` de las etiquetas 2 a 5 (usa `.loc`).
2. Filtra los empleados con `nivel == 4` y `salario > 3000`; muestra `edad`, `salario` y `nivel`.

**Resultado esperado:** 4 filas en el apartado 1.


In [ ]:
print(empleados_idx.loc[2:5, ["edad", "salario"]])

filtro = (empleados["nivel"] == 4) & (empleados["salario"] > 3000)
print(empleados.loc[filtro, ["edad", "salario", "nivel"]])


## Bloque 3 — Transformaciones

1. Crea `salario_miles = salario / 1000`.
2. Crea `tramo_horas` con `pd.cut` sobre `horas`: bins `[0, 35, 42, 60]` y etiquetas `["parcial", "completa", "extra"]`.
3. Cuenta cuántos empleados hay en cada tramo con `value_counts()`.

**Resultado esperado:** tres tramos; la suma de los recuentos es 1000.


In [ ]:
empleados["salario_miles"] = empleados["salario"] / 1000
empleados["tramo_horas"] = pd.cut(
    empleados["horas"], bins=[0, 35, 42, 60], labels=["parcial", "completa", "extra"]
)
print(empleados["tramo_horas"].value_counts())


## Bloque 4 — `groupby`

1. Si no existe, crea `senior = np.where(experiencia >= 10, "sí", "no")`.
2. Calcula la media de `salario` por `senior`.
3. Para `satisfaccion`, muestra `size()` y `count()` por `nivel` y explica la diferencia en un comentario.

**Resultado esperado:** `size()` suma 1000 y `count()` suma 985.


In [ ]:
if "senior" not in empleados:
    empleados["senior"] = np.where(empleados["experiencia"] >= 10, "sí", "no")

print(empleados.groupby("senior")["salario"].mean())
print(empleados.groupby("nivel")["satisfaccion"].size())
print(empleados.groupby("nivel")["satisfaccion"].count())
# size() cuenta filas (1000 en total) y count() ignora NaN (985 en total).


## Bloque 5 — Ausentes

1. Cuenta los ausentes por columna con `isna().sum()`.
2. Crea `satisfaccion_imp` imputando con la mediana y comprueba que ya no quedan nulos.

**Resultado esperado:** solo `satisfaccion` tiene 15; tras imputar, 0.


In [ ]:
print(empleados.isna().sum())
empleados["satisfaccion_imp"] = empleados["satisfaccion"].fillna(
    empleados["satisfaccion"].median()
)
print(empleados[["satisfaccion", "satisfaccion_imp"]].isna().sum())


## Ticket de salida

Responde en la celda siguiente, en una frase cada una:

1. Diferencia entre `.loc` y `.iloc`.
2. Qué hace `groupby`.
3. Una comparación de una línea entre R y pandas para un filtro.


In [ ]:
# 1) .loc usa etiquetas (incluye el extremo) y .iloc usa posiciones (excluye el extremo).
# 2) groupby divide las filas en grupos, aplica un cálculo y combina el resultado.
# 3) R: empleados[empleados$salario > 3000, ] equivale a pandas: empleados[empleados["salario"] > 3000].
